# 🔙 Notebook 04: Backtracking
## Algoritmos y Estructuras de Datos
**Universidad de Talca** | Facultad de Ingeniería  
**Docente:** PhD. César Astudillo  
**Semestre:** _________ | **Fecha:** _________

---
> 🎯 *Este notebook está diseñado para ser ejecutado en clase de forma interactiva.  
> Ejecuta las celdas en orden de arriba hacia abajo.*

In [ ]:
# Verificación de dependencias — ejecutar primero
import sys
required = {
    'numpy': 'numpy',
    'matplotlib': 'matplotlib',
    'ipywidgets': 'ipywidgets',
}
for nombre, paquete in required.items():
    try:
        __import__(paquete)
        print(f"✅ {nombre} instalado correctamente")
    except ImportError:
        print(f"❌ {nombre} NO encontrado — instala con: pip install {paquete}")
print("\n🐍 Python", sys.version.split()[0], "| Todo listo para comenzar.")

## 🎯 Objetivos de Aprendizaje

Al finalizar esta sesión, el estudiante será capaz de:

1. **Comprender** el paradigma Backtracking: búsqueda exhaustiva organizada como árbol de decisiones con retroceso.
2. **Identificar** la función de poda y explicar por qué reduce el espacio de búsqueda sin perder soluciones.
3. **Implementar** en Python la generación de combinaciones de bits, permutaciones y el problema de las N-Reinas.
4. **Resolver** la Mochila 0/1 por backtracking y comparar el árbol explorado con las soluciones de NB 02 y NB 03.
5. **Distinguir** entre buscar una solución, todas las soluciones y la solución óptima con backtracking.

## 🌍 Motivación: Cuando Todo lo Demás Falla

Existen problemas donde:
- No podemos dividir en subproblemas independientes → D&V no aplica
- La decisión local óptima no garantiza el óptimo global → Greedy no aplica
- Los subproblemas solapados no tienen estructura explorable → DP no aplica o es intratable

En esos casos, la última opción es la **fuerza bruta inteligente**: explorar **todas** las soluciones posibles, pero con una estrategia para **no explorar caminos que nunca llegarán a una solución válida**.

Esa estrategia es el **Backtracking** (vuelta atrás).

> 📌 **Definición (del PDF):** El backtracking impone una estructura de **árbol** sobre el espacio de soluciones. Si en un nodo interno del árbol se detecta que **ningún descendiente** puede llevar a una solución válida, se **poda** esa rama y se retrocede al nodo anterior para explorar otra alternativa.

```
Ejemplo: ¿cómo coloco 4 reinas en un tablero 4×4 sin que se ataquen?

   Col: 1    2    3    4
F1:  Q──────────────────   ← Prueba reina en (1,1)
F2:     ✗    ✗    Q──     ← (2,1),(2,2) atacadas → probar (2,3)
F3:  ✗    ✗    ✗    ✗    ← todas atacadas → RETROCEDER a F2
F2:               Q──     ← probar (2,4)
F3:     Q──              ← (3,2) libre
F4:  ✗    ✗    ✗    ✗    ← todas atacadas → RETROCEDER
   ... y así hasta encontrar solución
```

> 🎙️ **[PAUSA PROFESOR]** *"¿Cuál es la diferencia entre backtracking y probar todas las combinaciones posibles por fuerza bruta?"*  
> *(Backtracking poda ramas enteras antes de explorarlas → en la práctica es mucho más rápido, aunque el peor caso sea igual.)*

## 📐 Teoría: El Paradigma Backtracking

### Estructura general (del PDF)

```
procedimiento backtrack(nodo_actual):
    SI nodo_actual ES solución:
        registrar/retornar solución
        RETORNAR
    PARA CADA extensión posible de nodo_actual:
        SI extensión NO viola restricciones (función de poda):  ← CLAVE
            aplicar extensión
            backtrack(nuevo_nodo)
            deshacer extensión                                  ← "vuelta atrás"
```

### Conceptos clave

> 📌 **Espacio de soluciones:** El conjunto de todas las posibles soluciones candidatas, organizado como un árbol donde la raíz es el estado inicial, los nodos internos son estados parciales y las **hojas** son las soluciones completas (válidas o no).

> 📌 **Función de poda:** Una condición que se evalúa en cada nodo interno. Si es `True`, la rama entera se descarta sin explorar. Una buena función de poda reduce drásticamente el número de nodos visitados.

> ⚠️ **Importante:** La poda NUNCA puede eliminar soluciones válidas — solo ramas que garantizadamente no contienen ninguna.

### Tres tipos de búsqueda

| Objetivo | Estrategia | Cuándo parar |
|----------|-----------|-------------|
| Encontrar **una** solución | Parar al hallar la primera hoja válida | Al primer éxito |
| Encontrar **todas** las soluciones | Siempre continuar hasta agotar el árbol | Al recorrer todo |
| Encontrar la solución **óptima** | Explorar todo, guardar el mejor | Al recorrer todo (o B&B, NB 05) |

> 💡 **Insight:** Backtracking es fuerza bruta **mejorada**. Sin poda explora $O(2^n)$ o $O(n!)$ nodos; con poda, en la práctica puede ser órdenes de magnitud más rápido — aunque el peor caso sigue siendo exponencial.

In [ ]:
# ── Árbol binario 000→111 del PDF ─────────────────────────────────────────
# Reproduce visualmente el diagrama del PDF S03_Diseño_de_Algoritmos.
# Muestra el árbol completo de combinaciones de n=3 bits.
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

try:
    import google.colab; EN_COLAB = True
except ImportError:
    EN_COLAB = False
if not EN_COLAB:
    try: get_ipython().run_line_magic('matplotlib', 'inline')
    except Exception: pass

# Paleta del curso
C_RAIZ    = '#2196F3'   # azul — raíz
C_INTERNO = '#90CAF9'   # azul claro — nodos internos
C_HOJA    = '#4CAF50'   # verde — hojas (soluciones)
C_TEXTO   = '#212121'
C_FONDO   = '#FAFAFA'
C_ARISTA0 = '#9C27B0'   # morado — rama '0'
C_ARISTA1 = '#FF9800'   # naranja — rama '1'

# Coordenadas del árbol — replicando el diagrama del PDF
# Nivel 0: raíz (vacío)
# Nivel 1: 0, 1
# Nivel 2: 00, 01, 10, 11
# Nivel 3: 000, 001, 010, 011, 100, 101, 110, 111  (hojas)

NODOS = [
    # (id, label, x, y, es_hoja)
    ('r',   '',     0.50, 3.5, False),   # raíz
    ('0',   '0',    0.25, 2.5, False),
    ('1',   '1',    0.75, 2.5, False),
    ('00',  '00',   0.125,1.5, False),
    ('01',  '01',   0.375,1.5, False),
    ('10',  '10',   0.625,1.5, False),
    ('11',  '11',   0.875,1.5, False),
    ('000', '000',  0.0625,0.5, True),
    ('001', '001',  0.1875,0.5, True),
    ('010', '010',  0.3125,0.5, True),
    ('011', '011',  0.4375,0.5, True),
    ('100', '100',  0.5625,0.5, True),
    ('101', '101',  0.6875,0.5, True),
    ('110', '110',  0.8125,0.5, True),
    ('111', '111',  0.9375,0.5, True),
]

ARISTAS = [
    # (padre_id, hijo_id, bit)
    ('r',  '0',   '0'), ('r',  '1',   '1'),
    ('0',  '00',  '0'), ('0',  '01',  '1'),
    ('1',  '10',  '0'), ('1',  '11',  '1'),
    ('00', '000', '0'), ('00', '001', '1'),
    ('01', '010', '0'), ('01', '011', '1'),
    ('10', '100', '0'), ('10', '101', '1'),
    ('11', '110', '0'), ('11', '111', '1'),
]

nodo_dict = {n[0]: n for n in NODOS}

fig, ax = plt.subplots(figsize=(14, 7))
ax.set_facecolor(C_FONDO); fig.patch.set_facecolor(C_FONDO)
ax.set_xlim(-0.02, 1.02); ax.set_ylim(0.0, 4.1)
ax.axis('off')

# Dibujar aristas con etiqueta del bit
for padre_id, hijo_id, bit in ARISTAS:
    px, py = nodo_dict[padre_id][2], nodo_dict[padre_id][3]
    hx, hy = nodo_dict[hijo_id][2],  nodo_dict[hijo_id][3]
    color_arista = C_ARISTA0 if bit == '0' else C_ARISTA1
    ax.plot([px, hx], [py - 0.18, hy + 0.18],
            color=color_arista, lw=1.8, zorder=1)
    mx, my = (px + hx) / 2, (py + hy) / 2
    ax.text(mx + 0.008, my, bit, fontsize=9, color=color_arista,
            fontweight='bold', ha='left', va='center', zorder=4)

# Dibujar nodos
for nid, label, x, y, es_hoja in NODOS:
    if nid == 'r':
        color = C_RAIZ
        radio = 0.022
        display_label = 'ε'
    elif es_hoja:
        color = C_HOJA
        radio = 0.030
        display_label = label
    else:
        color = C_INTERNO
        radio = 0.022
        display_label = label

    circ = plt.Circle((x, y), radio, color=color,
                       ec=C_TEXTO, lw=1.2, zorder=2)
    ax.add_patch(circ)
    fs = 8 if es_hoja else 7
    fw = 'bold'
    ax.text(x, y, display_label, ha='center', va='center',
            fontsize=fs, color='white', fontweight=fw, zorder=3)

# Etiquetas de nivel
niveles = [(3.5, 'Nivel 0\n(raíz ε)'),
           (2.5, 'Nivel 1\n(1 bit)'),
           (1.5, 'Nivel 2\n(2 bits)'),
           (0.5, 'Nivel 3\n(hojas — 3 bits)')]
for ypos, etq in niveles:
    ax.text(-0.01, ypos, etq, ha='right', va='center',
            fontsize=8, color='#555', style='italic')

# Leyenda
leyenda = [
    mpatches.Patch(color=C_RAIZ,     label='Raíz (estado inicial)'),
    mpatches.Patch(color=C_INTERNO,  label='Nodo interno (decisión parcial)'),
    mpatches.Patch(color=C_HOJA,     label='Hoja (solución completa — 3 bits)'),
    mpatches.Patch(color=C_ARISTA0,  label="Rama '0'"),
    mpatches.Patch(color=C_ARISTA1,  label="Rama '1'"),
]
ax.legend(handles=leyenda, loc='upper right', fontsize=8,
          framealpha=0.9, edgecolor='#ccc')

ax.set_title(
    'Árbol de combinaciones de n=3 bits (000 → 111)\n'
    'Reproducción del diagrama del PDF S03_Diseño_de_Algoritmos\n'
    f'Total de hojas: 2³ = 8   |   Nodos internos: 7   |   Total: 15 nodos',
    fontsize=11, fontweight='bold', color=C_TEXTO
)
plt.tight_layout()
plt.show()
print("\n💡 Cada hoja representa una combinación de 3 bits.")
print("   Para n bits: 2ⁿ hojas → O(2ⁿ) complejidad en el peor caso.")
print("   La poda elimina ramas enteras SIN explorar sus hojas.")

In [ ]:
# ── Generación de todas las combinaciones de n bits — con backtracking ────

def generar_bits(n: int, verbose: bool = False) -> list:
    """
    Genera todas las combinaciones de n bits usando backtracking.
    Recorre el árbol del diagrama del PDF de izquierda a derecha.

    Parámetros:
        n (int):       número de bits
        verbose (bool): si True, muestra cada paso de exploración

    Retorna:
        list: lista de strings (cada string es una combinación de n bits)

    Complejidad:
        Temporal: O(2ⁿ · n) — 2ⁿ hojas × n pasos por hoja
        Espacial: O(n)       — profundidad máxima del árbol

    Ejemplo:
        >>> generar_bits(3)
        ['000', '001', '010', '011', '100', '101', '110', '111']
    """
    soluciones = []
    nodos_visitados = [0]   # contador para demostrar el trabajo realizado

    def bt(actual: list):
        """Función de backtracking — actual = lista de bits elegidos hasta ahora."""
        nodos_visitados[0] += 1

        if verbose:
            prefijo = '  ' * len(actual)
            print(f"{prefijo}→ Nodo: {''.join(str(b) for b in actual) or 'ε'}")

        # Caso base: hemos elegido n bits → solución completa
        if len(actual) == n:
            soluciones.append(''.join(str(b) for b in actual))
            return

        # Extensiones: agregar 0 o agregar 1
        for bit in [0, 1]:
            actual.append(bit)     # aplicar extensión
            bt(actual)             # explorar recursivamente
            actual.pop()           # deshacer extensión ← vuelta atrás

    bt([])
    return soluciones


# ── Demo ──────────────────────────────────────────────────────────────────
print("Árbol de n=3 bits (pasos de exploración):")
resultado = generar_bits(3, verbose=True)
print(f"\nSoluciones (hojas): {resultado}")
print(f"Total: {len(resultado)} = 2³")

print("\nConteo por n:")
for n_val in range(1, 8):
    sols = generar_bits(n_val)
    print(f"  n={n_val}: {len(sols):4d} combinaciones = 2^{n_val}")

In [ ]:
# ── Generación de todas las permutaciones de [1..n] ──────────────────────

def generar_permutaciones(elementos: list, verbose: bool = False) -> list:
    """
    Genera todas las permutaciones de 'elementos' usando backtracking.
    Usa un conjunto de elementos disponibles para la función de poda.

    Parámetros:
        elementos (list): lista de elementos a permutar
        verbose (bool):   si True, imprime cada permutación encontrada

    Retorna:
        list: lista de listas (cada sublista es una permutación)

    Complejidad:
        Temporal: O(n! · n) — n! hojas × n pasos por hoja
        Espacial: O(n)
    """
    soluciones = []

    def bt(actual: list, disponibles: list):
        """actual = permutación parcial; disponibles = elementos no usados aún."""
        # Caso base: hemos asignado todos los elementos
        if not disponibles:
            soluciones.append(actual.copy())
            if verbose:
                print(f"  ✅ Permutación: {actual}")
            return

        # Extensión: elegir cualquier elemento disponible como siguiente
        for elem in disponibles:
            nuevos_disp = [x for x in disponibles if x != elem]  # remover el elegido
            actual.append(elem)          # aplicar
            bt(actual, nuevos_disp)      # explorar
            actual.pop()                 # deshacer ← vuelta atrás

    bt([], elementos)
    return soluciones


# ── Demo ──────────────────────────────────────────────────────────────────
import math
for n_val in [2, 3, 4]:
    perms = generar_permutaciones(list(range(1, n_val + 1)))
    print(f"n={n_val}: {len(perms)} permutaciones = {n_val}! = {math.factorial(n_val)}")
    if n_val <= 3:
        print(f"       {perms}")

print("\n🔍 Todas las permutaciones de [1,2,3]:")
generar_permutaciones([1, 2, 3], verbose=True)
print()
print("💡 n=10 → 10! = 3 628 800 permutaciones.")
print("   n=12 → 12! ≈ 479 millones. El crecimiento factorial hace")
print("   que la poda sea CRÍTICA para n grandes.")

In [ ]:
# ── Problema de las N-Reinas ──────────────────────────────────────────────
# Colocar N reinas en un tablero N×N sin que ninguna se ataque.
# La función de poda verifica ataques en columna y diagonales.

def n_reinas(N: int, encontrar_todas: bool = False) -> list:
    """
    Resuelve el problema de las N-Reinas con backtracking.

    Parámetros:
        N (int):             tamaño del tablero (N×N)
        encontrar_todas (bool): si True, devuelve todas las soluciones;
                                si False, solo la primera encontrada

    Retorna:
        list: lista de soluciones, cada solución es una lista de N enteros
              donde sol[i] = columna de la reina en la fila i (0-indexed)

    Complejidad:
        Temporal: O(N!) en el peor caso — con poda mucho mejor en práctica
        Espacial: O(N)  — pila de recursión + estado actual
    """
    soluciones = []
    nodos_explorados = [0]
    nodos_podados    = [0]

    def es_valido(tablero: list, fila: int, col: int) -> bool:
        """Función de poda: ¿podemos colocar una reina en (fila, col)?"""
        for f_ant in range(fila):
            c_ant = tablero[f_ant]
            # Misma columna
            if c_ant == col:
                return False
            # Diagonal
            if abs(c_ant - col) == abs(f_ant - fila):
                return False
        return True

    def bt(tablero: list, fila: int):
        """tablero[i] = columna de la reina en la fila i."""
        nodos_explorados[0] += 1

        # Caso base: todas las filas llenadas → solución válida
        if fila == N:
            soluciones.append(tablero.copy())
            return

        # Extensión: intentar cada columna en la fila actual
        for col in range(N):
            if es_valido(tablero, fila, col):   # poda: solo si no hay ataques
                tablero.append(col)             # colocar reina
                bt(tablero, fila + 1)           # explorar siguiente fila
                tablero.pop()                   # quitar reina ← vuelta atrás
            else:
                nodos_podados[0] += 1

            if not encontrar_todas and soluciones:
                return  # encontramos la primera → parar

    bt([], 0)
    return soluciones, nodos_explorados[0], nodos_podados[0]


# ── Demo ──────────────────────────────────────────────────────────────────
print("Estadísticas del problema N-Reinas:")
print(f"{'N':>4} {'Soluciones':>12} {'Nodos expl.':>14} {'Podados':>10}")
print("-" * 44)
for N_val in range(4, 11):
    sols, expl, pod = n_reinas(N_val, encontrar_todas=True)
    print(f"{N_val:>4} {len(sols):>12} {expl:>14,} {pod:>10,}")

print("\n🔍 Una solución para N=8 (la clásica):")
sols8, _, _ = n_reinas(8, encontrar_todas=False)
sol = sols8[0]
print("   Tablero (Q=reina, .=vacío):")
for fila, col in enumerate(sol):
    linea = '   ' + ' '.join('♛' if c == col else '·' for c in range(8))
    print(linea)

In [ ]:
# Visualización del tablero N-Reinas + árbol de exploración parcial
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

def visualizar_tablero_reinas(solucion: list, titulo: str = ''):
    """Dibuja el tablero de ajedrez con la solución de N-Reinas."""
    N = len(solucion)
    fig, ax = plt.subplots(figsize=(min(N, 8), min(N, 8)))
    fig.patch.set_facecolor('#FAFAFA')

    # Tablero de ajedrez
    for fila in range(N):
        for col in range(N):
            color = '#F0D9B5' if (fila + col) % 2 == 0 else '#B58863'
            ax.add_patch(plt.Rectangle((col, N - 1 - fila), 1, 1,
                                        facecolor=color, edgecolor='#212121', lw=0.5))

    # Reinas
    for fila, col in enumerate(solucion):
        ax.text(col + 0.5, N - 1 - fila + 0.5, '♛',
                ha='center', va='center',
                fontsize=int(220 / N), color='#2196F3')

    # Etiquetas de filas/columnas
    for i in range(N):
        ax.text(-0.3, N - 1 - i + 0.5, str(i+1),
                ha='center', va='center', fontsize=8, color='#555')
        ax.text(i + 0.5, -0.3, str(i+1),
                ha='center', va='center', fontsize=8, color='#555')

    ax.set_xlim(-0.5, N)
    ax.set_ylim(-0.5, N)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(titulo or f'N-Reinas (N={N}) — solución: {solucion}',
                 fontsize=10, fontweight='bold', color='#212121')
    plt.tight_layout()
    plt.show()


# Mostrar todas las soluciones de N=4 y la primera de N=6, N=8
print("Todas las soluciones para N=4:")
sols4, expl4, pod4 = n_reinas(4, encontrar_todas=True)
print(f"  {len(sols4)} soluciones | {expl4} nodos explorados | {pod4} podados")
for i, s in enumerate(sols4):
    visualizar_tablero_reinas(s, f'N=4 — Solución {i+1}/{len(sols4)}: columnas={s}')

print("\nPrimera solución para N=8:")
sols8_u, e8, p8 = n_reinas(8, encontrar_todas=False)
print(f"  Nodos explorados: {e8:,} | Podados: {p8:,}")
visualizar_tablero_reinas(sols8_u[0], f'N=8 — Primera solución encontrada')

In [ ]:
# ── Mochila 0/1 por Backtracking ──────────────────────────────────────────
# Mismos datos que NB 02 (Greedy) y NB 03 (DP)
# Permite comparar los 3 enfoques al mismo problema.

# Datos del PDF (idénticos a NB 02, 03, 05)
OBJETOS = [
    {'nombre': 'Objeto 1', 'valor': 20, 'peso': 50},
    {'nombre': 'Objeto 2', 'valor': 24, 'peso': 100},
    {'nombre': 'Objeto 3', 'valor': 55, 'peso': 150},
    {'nombre': 'Objeto 4', 'valor': 40, 'peso': 200},
    {'nombre': 'Objeto 5', 'valor': 70, 'peso': 250},
]
CAPACIDAD_W = 300


def mochila_backtracking(objetos: list, capacidad: int) -> dict:
    """
    Resuelve la Mochila 0/1 usando backtracking exhaustivo.
    Explora todas las combinaciones de incluir/excluir cada objeto.

    Parámetros:
        objetos (list):  lista de dicts con 'nombre', 'valor', 'peso'
        capacidad (int): peso máximo de la mochila

    Retorna:
        dict: {'valor_optimo', 'seleccion', 'nodos_explorados', 'nodos_podados'}

    Complejidad:
        Temporal: O(2ⁿ) peor caso — cada objeto se incluye o excluye
        Espacial: O(n)  — profundidad de la recursión
    """
    n = len(objetos)
    mejor = {'valor': 0, 'seleccion': [], 'nodos_explorados': 0, 'nodos_podados': 0}

    def bt(idx: int, peso_actual: int, valor_actual: int, incluidos: list):
        """
        idx           = índice del objeto a decidir ahora
        peso_actual   = peso acumulado de los objetos ya incluidos
        valor_actual  = valor acumulado de los objetos ya incluidos
        incluidos     = lista de objetos incluidos hasta ahora
        """
        mejor['nodos_explorados'] += 1

        # Actualizar la mejor solución si la actual es mejor
        if valor_actual > mejor['valor']:
            mejor['valor']     = valor_actual
            mejor['seleccion'] = incluidos.copy()

        # Caso base: hemos decidido sobre todos los objetos
        if idx == n:
            return

        # Poda: si el objeto actual ya no cabe, ningún objeto posterior
        # (los objetos están ordenados por peso asc, pero aquí no los ordenamos;
        #  la poda simple verifica si el siguiente cabe)
        obj = objetos[idx]

        # Rama 1: INCLUIR el objeto idx (si cabe)
        if peso_actual + obj['peso'] <= capacidad:
            incluidos.append(obj['nombre'])
            bt(idx + 1, peso_actual + obj['peso'],
               valor_actual + obj['valor'], incluidos)
            incluidos.pop()   # ← vuelta atrás
        else:
            mejor['nodos_podados'] += 1   # poda: el objeto no cabe

        # Rama 2: EXCLUIR el objeto idx (siempre se puede)
        bt(idx + 1, peso_actual, valor_actual, incluidos)

    bt(0, 0, 0, [])
    return {
        'valor_optimo':     mejor['valor'],
        'seleccion':        mejor['seleccion'],
        'nodos_explorados': mejor['nodos_explorados'],
        'nodos_podados':    mejor['nodos_podados'],
    }


# ── Demo ──────────────────────────────────────────────────────────────────
print("Mochila 0/1 — Backtracking (datos del PDF, mismos que NB 02 y NB 03)")
print(f"Objetos: {[(o['nombre'], o['valor'], o['peso']) for o in OBJETOS]}")
print(f"Capacidad W = {CAPACIDAD_W}\n")

res_bt = mochila_backtracking(OBJETOS, CAPACIDAD_W)
print(f"✅ Resultado Backtracking:")
print(f"   Valor óptimo:      {res_bt['valor_optimo']}")
print(f"   Selección:         {res_bt['seleccion']}")
print(f"   Nodos explorados:  {res_bt['nodos_explorados']}")
print(f"   Nodos podados:     {res_bt['nodos_podados']}")
print(f"   Total posible sin poda: 2^{len(OBJETOS)} = {2**len(OBJETOS)}")
print()
print("📊 Comparación de los 3 enfoques (mismos datos):")
print(f"   NB 02 — Greedy (ratio v/w):  valor = ? (ejecuta NB 02 para ver)")
print(f"   NB 03 — DP:                  valor = 94 (óptimo garantizado)")
print(f"   NB 04 — Backtracking:        valor = {res_bt['valor_optimo']} (óptimo)")
print(f"   → Backtracking y DP dan el mismo resultado óptimo.")
print(f"   → Backtracking exploró {res_bt['nodos_explorados']} nodos vs DP que llenó una tabla.")

In [ ]:
# Animación: árbol de backtracking para N-Reinas con N=4
# Muestra nodos explorados (azul), podados (rojo) y soluciones (verde)
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as mpatches
from IPython.display import HTML, display

C_ACTIVO  = '#2196F3'   # azul
C_PODADO  = '#F44336'   # rojo
C_SOL     = '#4CAF50'   # verde
C_FONDO   = '#FAFAFA'
C_TEXTO   = '#212121'

def animar_bt_bits(n_bits=3, intervalo_ms=500):
    """
    Anima el recorrido del árbol de backtracking para combinaciones de bits.
    Muestra cada nodo visitado y el estado final de las hojas.
    """
    # Construir la secuencia de visitas
    visitas = []   # lista de (prefijo_actual, tipo) donde tipo='interno'|'hoja'

    def bt_registro(actual):
        visitas.append((''.join(str(b) for b in actual) or 'ε', 'hoja' if len(actual)==n_bits else 'interno'))
        if len(actual) == n_bits:
            return
        for bit in [0, 1]:
            actual.append(bit)
            bt_registro(actual)
            actual.pop()

    bt_registro([])

    # Todas las posiciones del árbol (igual que el diagrama estático)
    pos = {}
    pos['ε'] = (0.5, n_bits)
    for nivel in range(1, n_bits + 1):
        n_nodos = 2**nivel
        for i in range(n_nodos):
            etq = format(i, f'0{nivel}b')
            pos[etq] = ((i + 0.5) / n_nodos, n_bits - nivel)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.set_facecolor(C_FONDO); fig.patch.set_facecolor(C_FONDO)
    ax.set_xlim(0, 1); ax.set_ylim(-0.3, n_bits + 0.5)
    ax.axis('off')

    # Dibujar todas las aristas (fijo)
    for nivel in range(n_bits):
        for i in range(2**nivel):
            etq_p = format(i, f'0{nivel}b') if nivel > 0 else 'ε'
            for bit in [0, 1]:
                hijo = (format(i, f'0{nivel}b') if nivel > 0 else '') + str(bit)
                if etq_p in pos and hijo in pos:
                    ax.plot([pos[etq_p][0], pos[hijo][0]],
                            [pos[etq_p][1] - 0.12, pos[hijo][1] + 0.12],
                            color='#BDBDBD', lw=1, zorder=1)

    # Círculos de nodos (dinámicos)
    circulos = {}
    for etq, (x, y) in pos.items():
        circ = plt.Circle((x, y), 0.025, color='#E0E0E0', ec='#9E9E9E', lw=1, zorder=2)
        ax.add_patch(circ)
        ax.text(x, y, etq, ha='center', va='center',
                fontsize=6.5, color='#757575', zorder=3)
        circulos[etq] = circ

    titulo = ax.set_title('Backtracking — árbol de n=3 bits',
                           fontsize=10, fontweight='bold', color=C_TEXTO)
    paso_txt = ax.text(0.98, 0.97, '', transform=ax.transAxes,
                       ha='right', va='top', fontsize=9, color=C_TEXTO)

    def actualizar(fi):
        etq, tipo = visitas[fi]
        color = C_SOL if tipo == 'hoja' else C_ACTIVO
        if etq in circulos:
            circulos[etq].set_facecolor(color)
            circulos[etq].set_edgecolor(C_TEXTO)
        titulo.set_text(f'Visitando nodo: "{etq}" ({tipo})')
        paso_txt.set_text(f'Paso {fi+1}/{len(visitas)}')

    anim = animation.FuncAnimation(fig, actualizar, frames=len(visitas),
                                    interval=intervalo_ms, repeat=False, blit=False)
    plt.tight_layout()
    plt.close(fig)
    return anim

print("Animación: recorrido del árbol backtracking para n=3 bits")
anim = animar_bt_bits(n_bits=3, intervalo_ms=450)
display(HTML(anim.to_jshtml()))

## 📈 Análisis de Complejidad

| Problema | Sin poda | Con poda (práctica) | Espacio |
|----------|---------|--------------------|---------|
| Combinaciones de $n$ bits | $O(2^n)$ | $O(2^n)$ — no hay poda posible | $O(n)$ |
| Permutaciones de $n$ elementos | $O(n!)$ | $O(n!)$ en el peor caso | $O(n)$ |
| N-Reinas | $O(N^N)$ teórico | Mucho mejor — ej. N=8: solo 15 720 nodos vs $8^8$=16M | $O(N)$ |
| Mochila 0/1 BT | $O(2^n)$ | Depende de la distribución de pesos | $O(n)$ |

### Comparación de la Mochila 0/1 — tres enfoques

| Enfoque | Temporal | Espacial | Garantía de óptimo | NB |
|---------|---------|---------|-------------------|----|n
| Greedy (ratio v/w) | $O(n \log n)$ | $O(n)$ | ❌ Solo fraccionaria | 02 |
| Programación Dinámica | $O(n \cdot W)$ | $O(n \cdot W)$ | ✅ Siempre | 03 |
| Backtracking | $O(2^n)$ peor | $O(n)$ | ✅ Siempre | 04 |
| Branch & Bound | Mejor que BT | $O(n)$–$O(2^n)$ | ✅ Siempre | 05 |

> ⚠️ **Importante:** Backtracking usa $O(n)$ de espacio (solo la pila de recursión), a diferencia de DP que usa $O(n \cdot W)$. Esto puede ser relevante cuando $W$ es muy grande.

> 💡 **Insight:** La calidad de la función de poda determina la eficiencia práctica de backtracking. Una mala poda → exponencial. Una buena poda → puede ser casi lineal para ciertos problemas.

In [ ]:
# Widget interactivo: N-Reinas con visualización del tablero
# Auto-contenido — no necesita celdas anteriores.
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt

def _n_reinas_widget(N, encontrar_todas=False):
    """Versión compacta de N-Reinas para el widget."""
    soluciones = []
    explorados = [0]; podados = [0]
    def es_valido(t, f, c):
        for fa in range(f):
            if t[fa] == c or abs(t[fa]-c) == abs(fa-f): return False
        return True
    def bt(t, f):
        explorados[0] += 1
        if f == N:
            soluciones.append(t.copy()); return
        for c in range(N):
            if es_valido(t, f, c):
                t.append(c); bt(t, f+1); t.pop()
            else:
                podados[0] += 1
            if not encontrar_todas and soluciones: return
    bt([], 0)
    return soluciones, explorados[0], podados[0]

def _dibujar_tablero(ax, sol, titulo=''):
    N = len(sol)
    for f in range(N):
        for c in range(N):
            color = '#F0D9B5' if (f+c)%2==0 else '#B58863'
            ax.add_patch(plt.Rectangle((c, N-1-f), 1, 1,
                         facecolor=color, edgecolor='#555', lw=0.5))
    for f, c in enumerate(sol):
        ax.text(c+0.5, N-1-f+0.5, '♛', ha='center', va='center',
                fontsize=int(180/N), color='#2196F3')
    ax.set_xlim(0, N); ax.set_ylim(0, N)
    ax.set_aspect('equal'); ax.axis('off')
    if titulo: ax.set_title(titulo, fontsize=8)

# Controles
n_slider   = widgets.IntSlider(value=6, min=4, max=10, description='N (tablero):',
                                style={'description_width':'initial'},
                                layout=widgets.Layout(width='350px'))
todas_chk  = widgets.Checkbox(value=False, description='Mostrar TODAS las soluciones',
                               style={'description_width':'initial'})
boton      = widgets.Button(description='▶ Resolver', button_style='primary')
salida     = widgets.Output()

def al_resolver(b):
    with salida:
        salida.clear_output(wait=True)
        N         = n_slider.value
        todas     = todas_chk.value
        sols, exp, pod = _n_reinas_widget(N, encontrar_todas=todas)

        print(f"N={N} — {'Todas las' if todas else 'Primera'} solución(es)")
        print(f"  Soluciones encontradas: {len(sols)}")
        print(f"  Nodos explorados:  {exp:,}")
        print(f"  Nodos podados:     {pod:,}")
        print(f"  Sin poda (N^N):    {N**N:,}")

        n_mostrar = min(len(sols), 4 if todas else 1)
        if n_mostrar == 0:
            print("  (Sin soluciones)")
            return
        cols_fig = min(n_mostrar, 4)
        fig, axes = plt.subplots(1, cols_fig, figsize=(cols_fig * (N*0.5+0.5), N*0.5+0.5))
        if cols_fig == 1: axes = [axes]
        fig.patch.set_facecolor('#FAFAFA')
        for i, ax in enumerate(axes):
            _dibujar_tablero(ax, sols[i], f'Sol {i+1}')
        plt.suptitle(f'N-Reinas (N={N}) — mostrando {cols_fig}/{len(sols)} soluciones',
                     fontweight='bold', fontsize=9)
        plt.tight_layout()
        plt.show()

boton.on_click(al_resolver)
display(widgets.VBox([
    widgets.HTML('<h4 style="color:#212121">♛ Visualizador de N-Reinas (Backtracking)</h4>'),
    n_slider, todas_chk, boton, salida
]))

## 🧪 Ejercicio 1: Todas las Permutaciones ⭐

**Descripción:** Genera todas las permutaciones de una lista de $n$ elementos **distintos** usando backtracking.
Para $n=4$, muestra cuántas permutaciones hay y listarlas todas.

**Entrada:** lista de elementos (hasta $n=6$ para no sobrecargar la salida)

**Salida:** lista de todas las permutaciones (lista de listas)

**Ejemplo:**
```
Entrada: [1, 2, 3]
Salida:  [[1,2,3],[1,3,2],[2,1,3],[2,3,1],[3,1,2],[3,2,1]]
```

**Restricciones:** los elementos de la lista son distintos entre sí  
**Complejidad esperada:** $O(n!)$ en tiempo, $O(n)$ en espacio

In [ ]:
def todas_las_permutaciones(elementos: list) -> list:
    """
    Genera todas las permutaciones de 'elementos' usando backtracking.

    Parámetros:
        elementos (list): lista de elementos distintos

    Retorna:
        list: lista de listas — cada sublista es una permutación

    Complejidad:
        Temporal: O(n! · n)
        Espacial: O(n)
    """
    # Tu código aquí
    # Pista: define una función interna bt(actual, disponibles)
    # Cuando disponibles está vacío, actual es una permutación completa.
    pass

In [ ]:
def verificar_ejercicio_1(fn):
    """Ejecuta casos de prueba para todas_las_permutaciones."""
    import time, math
    casos = [
        ([],       1,  'Lista vacía — 1 permutación (la vacía)'),
        ([1],      1,  'Un elemento — 1! = 1'),
        ([1,2],    2,  'Dos elementos — 2! = 2'),
        ([1,2,3],  6,  'Tres elementos — 3! = 6'),
        ([1,2,3,4],24, 'Cuatro elementos — 4! = 24'),
    ]
    aprobados = 0
    for elems, esperado, descripcion in casos:
        t0 = time.perf_counter()
        try:
            resultado = fn(elems)
            t1 = time.perf_counter()
            # Verificar cantidad y que no haya duplicados
            sin_dup = len(set(tuple(p) for p in resultado))
            if len(resultado) == esperado and sin_dup == esperado:
                print(f"  ✅ {descripcion} ({(t1-t0)*1000:.2f}ms)")
                aprobados += 1
            else:
                print(f"  ❌ {descripcion}")
                print(f"     Esperado: {esperado} | Obtenido: {len(resultado)} | Únicos: {sin_dup}")
        except Exception as e:
            print(f"  💥 {descripcion} — Error: {e}")
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados==len(casos) else f'⚠️  {aprobados}/{len(casos)} casos correctos'}")

verificar_ejercicio_1(todas_las_permutaciones)

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# def todas_las_permutaciones(elementos):
#     """Genera todas las permutaciones con backtracking — O(n!)."""
#     soluciones = []
#     def bt(actual, disponibles):
#         if not disponibles:
#             soluciones.append(actual.copy())
#             return
#         for elem in disponibles:
#             nuevos = [x for x in disponibles if x != elem]
#             actual.append(elem)
#             bt(actual, nuevos)
#             actual.pop()     # ← vuelta atrás
#     bt([], elementos)
#     return soluciones

## 🧪 Ejercicio 2: N-Reinas — Una y Todas ⭐⭐

**Descripción:** Resuelve el problema de las N-Reinas usando backtracking.
Para $N=8$:
- Encuentra **una** solución
- Cuenta cuántas soluciones distintas existen en total

**Entrada:** entero $N$ (tamaño del tablero)

**Salida:** tupla `(una_solucion, total_soluciones)` donde `una_solucion` es una lista de $N$ enteros

**Ejemplo:**
```
Entrada: N = 4
Salida:  ([1, 3, 0, 2], 2)  — 2 soluciones para N=4
```

**Restricciones:** $4 \leq N \leq 10$  
**Complejidad esperada:** $O(N!)$ en el peor caso

In [ ]:
def resolver_n_reinas(N: int) -> tuple:
    """
    Resuelve el problema de las N-Reinas.
    Retorna UNA solución y el TOTAL de soluciones.

    Parámetros:
        N (int): tamaño del tablero N×N

    Retorna:
        tuple: (lista_una_solucion, total_soluciones_int)
               donde lista_una_solucion[i] = columna de la reina en fila i

    Complejidad:
        Temporal: O(N!) peor caso
        Espacial: O(N)
    """
    # Tu código aquí
    # Pista: usa la función es_valido(tablero, fila, col) para verificar ataques.
    # Para contar TODAS las soluciones: no pares al encontrar la primera.
    pass

In [ ]:
def verificar_ejercicio_2(fn):
    """Ejecuta casos de prueba para resolver_n_reinas."""
    import time
    # Número conocido de soluciones para N-Reinas
    soluciones_conocidas = {4:2, 5:10, 6:4, 7:40, 8:92, 9:352, 10:724}
    casos = list(soluciones_conocidas.items())
    aprobados = 0
    for N_val, total_esperado in casos:
        t0 = time.perf_counter()
        try:
            una_sol, total = fn(N_val)
            t1 = time.perf_counter()
            # Verificar: total correcto + solución válida
            valida = (len(una_sol) == N_val and
                      len(set(una_sol)) == N_val and
                      all(abs(una_sol[i]-una_sol[j]) != abs(i-j)
                          for i in range(N_val) for j in range(i+1, N_val)))
            if total == total_esperado and valida:
                print(f"  ✅ N={N_val}: {total} soluciones, solución válida ({(t1-t0)*1000:.0f}ms)")
                aprobados += 1
            else:
                print(f"  ❌ N={N_val}: esperado {total_esperado} soluciones, obtenido {total}")
                if not valida: print(f"     Solución inválida: {una_sol}")
        except Exception as e:
            print(f"  💥 N={N_val} — Error: {e}")
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados==len(casos) else f'⚠️  {aprobados}/{len(casos)} casos correctos'}")

verificar_ejercicio_2(resolver_n_reinas)

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# def resolver_n_reinas(N):
#     """N-Reinas con backtracking — devuelve una solución y el total."""
#     soluciones = []
#
#     def es_valido(tablero, fila, col):
#         for f_ant in range(fila):
#             c_ant = tablero[f_ant]
#             if c_ant == col or abs(c_ant - col) == abs(f_ant - fila):
#                 return False
#         return True
#
#     def bt(tablero, fila):
#         if fila == N:
#             soluciones.append(tablero.copy())
#             return
#         for col in range(N):
#             if es_valido(tablero, fila, col):
#                 tablero.append(col)
#                 bt(tablero, fila + 1)
#                 tablero.pop()     # ← vuelta atrás
#
#     bt([], 0)
#     return soluciones[0] if soluciones else [], len(soluciones)

## 🧪 Ejercicio 3: Sudoku Solver ⭐⭐⭐

**Descripción:** Implementa un resolvedor de Sudoku usando backtracking.
El tablero es una cuadrícula $9 \times 9$ donde:
- Los dígitos `1`–`9` no se repiten en ninguna fila
- Los dígitos `1`–`9` no se repiten en ninguna columna
- Los dígitos `1`–`9` no se repiten en ninguno de los 9 subcuadros $3 \times 3$
- El valor `0` representa una celda vacía

La función de poda verifica las tres restricciones antes de colocar un dígito.

**Entrada:** tablero como lista de 9 listas de 9 enteros (0=vacío)

**Salida:** tablero resuelto (lista de listas) o `None` si no tiene solución

**Complejidad esperada:** $O(9^m)$ donde $m$ = número de celdas vacías  
**Nivel CF equivalente:** ~1200–1400

In [ ]:
def resolver_sudoku(tablero: list) -> list:
    """
    Resuelve un Sudoku 9×9 usando backtracking.
    Las celdas vacías se representan con 0.

    Parámetros:
        tablero (list): lista de 9 listas de 9 enteros (0 = vacío)

    Retorna:
        list: tablero resuelto, o None si no tiene solución
              (modifica el tablero in-place y también lo retorna)

    Complejidad:
        Temporal: O(9^m) donde m = celdas vacías
        Espacial: O(m) — pila de recursión
    """
    # Tu código aquí
    # Pista:
    # 1. Encuentra la próxima celda vacía (valor 0).
    # 2. Prueba dígitos del 1 al 9.
    # 3. Antes de colocar, verifica fila, columna y subcuadro 3×3.
    # 4. Si ningún dígito funciona → retorna False (backtrack).
    pass

In [ ]:
def verificar_ejercicio_3(fn):
    """Ejecuta casos de prueba para resolver_sudoku."""
    import time, copy

    def es_valido_sudoku(t):
        """Verifica que un tablero resuelto sea válido."""
        digitos = set(range(1, 10))
        for i in range(9):
            if set(t[i]) != digitos: return False           # fila
            if set(t[r][i] for r in range(9)) != digitos: return False  # columna
        for br in range(3):
            for bc in range(3):
                bloque = set(t[br*3+r][bc*3+c] for r in range(3) for c in range(3))
                if bloque != digitos: return False
        return True

    # Tablero fácil
    facil = [
        [5,3,0, 0,7,0, 0,0,0],
        [6,0,0, 1,9,5, 0,0,0],
        [0,9,8, 0,0,0, 0,6,0],
        [8,0,0, 0,6,0, 0,0,3],
        [4,0,0, 8,0,3, 0,0,1],
        [7,0,0, 0,2,0, 0,0,6],
        [0,6,0, 0,0,0, 2,8,0],
        [0,0,0, 4,1,9, 0,0,5],
        [0,0,0, 0,8,0, 0,7,9]
    ]
    casos = [('Sudoku estándar', copy.deepcopy(facil))]

    aprobados = 0
    for nombre, t in casos:
        t0 = time.perf_counter()
        try:
            resuelto = fn(t)
            t1 = time.perf_counter()
            if resuelto is not None and es_valido_sudoku(resuelto):
                print(f"  ✅ {nombre} resuelto correctamente ({(t1-t0)*1000:.0f}ms)")
                aprobados += 1
                print("     Primeras 3 filas del resultado:")
                for fila in resuelto[:3]:
                    print(f"       {fila}")
            else:
                print(f"  ❌ {nombre} — solución inválida o None")
        except Exception as e:
            print(f"  💥 {nombre} — Error: {e}")
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados==len(casos) else f'⚠️  {aprobados}/{len(casos)} casos correctos'}")

verificar_ejercicio_3(resolver_sudoku)

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# def resolver_sudoku(tablero):
#     """Sudoku solver con backtracking — O(9^m)."""
#     def posible(t, fila, col, num):
#         """¿Se puede colocar 'num' en (fila, col)?"""
#         if num in t[fila]: return False                    # fila
#         if num in (t[r][col] for r in range(9)): return False  # columna
#         br, bc = (fila // 3) * 3, (col // 3) * 3
#         for r in range(br, br+3):
#             for c in range(bc, bc+3):
#                 if t[r][c] == num: return False            # subcuadro
#         return True
#
#     def bt():
#         for f in range(9):
#             for c in range(9):
#                 if tablero[f][c] == 0:            # celda vacía
#                     for num in range(1, 10):
#                         if posible(tablero, f, c, num):
#                             tablero[f][c] = num   # colocar
#                             if bt(): return True  # explorar
#                             tablero[f][c] = 0     # ← vuelta atrás
#                     return False    # ningún dígito funcionó — backtrack
#         return True  # no hay celdas vacías → resuelto
#
#     return tablero if bt() else None

## 🔬 Zona de Experimentación

Las siguientes celdas son tuyas para experimentar. Algunas sugerencias:
- ¿Qué pasa con el número de nodos explorados en N-Reinas si cambias el orden de exploración de columnas (de derecha a izquierda en vez de izquierda a derecha)?
- ¿Puedes modificar `generar_bits` para que genere combinaciones de `k` elementos de un conjunto de `n` (ej. todas las combinaciones de 3 elementos de {0,1,2,3,4})?
- Prueba `mochila_backtracking` agregando una poda adicional: si el valor restante máximo posible (incluir todos los objetos que quedan) no puede superar el mejor encontrado → podar.

In [ ]:
# Espacio libre para experimentar
# Sugerencia: genera todas las combinaciones de k=2 elementos de [1,2,3,4,5]

In [ ]:
# Espacio libre para experimentar
# Sugerencia: agrega una poda de cota superior a mochila_backtracking
# y compara cuántos nodos se podan adicionalmente

In [ ]:
# Autoevaluación — 4 preguntas sobre Backtracking
import ipywidgets as widgets
from IPython.display import display

preguntas = [
    {
        'pregunta': '1. ¿Qué garantía da la función de poda en Backtracking?',
        'opciones': [
            'Que el algoritmo termina en tiempo polinomial',
            'Que no se exploran ramas que garantizadamente no llevan a solución válida, sin perder soluciones',
            'Que siempre se encuentra la solución óptima',
            'Que el número de nodos explorados es igual al número de soluciones'
        ],
        'correcta': 1,
        'explicacion': 'La poda elimina ramas donde NINGÚN descendiente puede ser solución válida. Esto reduce el trabajo sin comprometer la completitud: si hay solución en esa rama, la poda no debe eliminarla.'
    },
    {
        'pregunta': '2. El árbol de n=3 bits tiene 15 nodos en total. ¿Cuántos son hojas?',
        'opciones': ['4', '6', '8', '15'],
        'correcta': 2,
        'explicacion': 'Las hojas son las combinaciones completas de 3 bits: 000, 001, 010, 011, 100, 101, 110, 111 = 2³ = 8 hojas. Los 7 nodos restantes son internos (raíz + 2 del nivel 1 + 4 del nivel 2).'
    },
    {
        'pregunta': '3. En el problema de las N-Reinas, ¿por qué es suficiente verificar ataques solo hacia ARRIBA (filas anteriores) y no también hacia abajo?',
        'opciones': [
            'Porque las reinas no atacan hacia abajo en el ajedrez',
            'Porque backtracking coloca reinas fila a fila: las filas inferiores aún no tienen reinas al momento de verificar',
            'Porque la verificación hacia abajo es O(N²) y hacia arriba es O(N)',
            'No es suficiente, también hay que verificar hacia abajo'
        ],
        'correcta': 1,
        'explicacion': 'Backtracking llena el tablero de arriba hacia abajo (fila 0, 1, 2...). Cuando verificamos si podemos colocar una reina en la fila i, las filas i+1, i+2... todavía están vacías. Solo necesitamos verificar conflictos con filas 0..i-1.'
    },
    {
        'pregunta': '4. ¿Cuál es la diferencia clave entre Backtracking y el Branch & Bound del Notebook 05?',
        'opciones': [
            'Backtracking usa DFS y Branch & Bound usa BFS',
            'Backtracking solo poda por inviabilidad; B&B también poda ramas subóptimas usando cotas de calidad',
            'Branch & Bound siempre encuentra más soluciones que Backtracking',
            'No hay diferencia: son el mismo algoritmo con distinto nombre'
        ],
        'correcta': 1,
        'explicacion': 'Backtracking poda cuando la solución parcial viola una restricción (inviable). B&B agrega una función de cota: si la MEJOR solución posible en esta rama es peor que la ya encontrada → también se poda. Esto permite descartar muchas más ramas en problemas de optimización.'
    }
]

def crear_quiz(preguntas):
    for i, p in enumerate(preguntas):
        radio  = widgets.RadioButtons(options=p['opciones'], description=f'P{i+1}:',
                                       style={'description_width':'initial'},
                                       layout={'width':'max-content'})
        boton  = widgets.Button(description='Verificar', button_style='info')
        salida = widgets.Output()
        label  = widgets.HTML(f'<b>{p["pregunta"]}</b>')
        def verificar(b, r=radio, out=salida,
                      c=p['correcta'], ex=p['explicacion'], opts=p['opciones']):
            with out:
                out.clear_output()
                if r.value == opts[c]:
                    print(f'✅ ¡Correcto! {ex}')
                else:
                    print(f'❌ No exactamente. Respuesta: "{opts[c]}"\n   {ex}')
        boton.on_click(verificar)
        display(widgets.VBox([label, radio, boton, salida]))
        print('─' * 65)

crear_quiz(preguntas)

## 📚 Lecturas Recomendadas y Práctica

### Textbooks

| Libro | Edición | Capítulo | Tema |
|-------|---------|----------|------|
| Skiena | 3ª ed. | **Cap. 9** — Combinatorial Search | Backtracking extenso: N-Reinas, Sudoku, ciclos Hamiltonianos |
| Cormen et al. (CLRS) | 4ª ed. | Apéndice / Cap. 34 (NP) | No tiene capítulo dedicado; ver Skiena para backtracking |
| Kleinberg & Tardos | 1ª ed. | **Cap. 7** (parcial) | Búsqueda exhaustiva en contexto de optimización |

### Recursos Gratuitos

- 🌐 [VisuAlgo — Backtracking](https://visualgo.net/en) — buscar N-Queens y Sudoku
- 📖 [CP-Algorithms — Backtracking](https://cp-algorithms.com/algebra/all-submasks.html) — enumeración de submáscaras
- 🌐 [USACO Guide — Complete Search](https://usaco.guide/bronze/intro-complete?lang=py) — guía con Python 3

### Práctica en Codeforces (soporta Python 3)

> 🔍 **Cómo filtrar:** [codeforces.com/problemset](https://codeforces.com/problemset) → Tag: `brute force`

| # | Criterio de búsqueda | Rating | Por qué es útil |
|---|---------------------|--------|----------------|
| 1 | Tag `brute force` + Rating 800 | ⭐ 800 | Enumeración simple — todos los subconjuntos o permutaciones |
| 2 | Tag `brute force` + Rating 1000 | ⭐⭐ 1000 | Requiere poda básica para no TLE |
| 3 | Tag `brute force` + Tag `backtracking` + Rating 1200 | ⭐⭐⭐ 1200 | Backtracking con restricciones |
| 4 | Tag `backtracking` + Rating 1400 | 🏆 1400+ | Backtracking avanzado con poda sofisticada |

> ⚠️ En Codeforces, el tag para backtracking básico es `brute force`. El tag `backtracking` es más raro y se usa para problemas donde la poda es explícitamente necesaria.

---

**Próximo notebook:** [05_ramificacion_y_poda.ipynb](05_ramificacion_y_poda.ipynb) — Backtracking + función de cota para podar ramas subóptimas.